In [3]:
import pandas as pd
import numpy as np
import re

# Load dataframe
df = pd.read_csv("found_ships_july.csv")

# Parse timestamps where possible (keep only time-of-day)
ts_parsed = pd.to_datetime(df["# Timestamp"], errors="coerce", infer_datetime_format=True)

# Build seconds-since-midnight series (float; will convert to int seconds)
seconds = pd.Series(np.nan, index=df.index, dtype=float)
mask_parsed = ts_parsed.notna()
if mask_parsed.any():
    parsed = ts_parsed[mask_parsed]
    seconds.loc[mask_parsed] = (
        parsed.dt.hour.astype(float) * 3600.0
        + parsed.dt.minute.astype(float) * 60.0
        + parsed.dt.second.astype(float)
        + parsed.dt.microsecond.astype(float) / 1e6
    )

# Fallback: extract HH:MM[:SS] with regex for unparsed rows
time_re = re.compile(r"(\d{1,2}):(\d{2})(?::(\d{2}))?")
for idx in seconds[seconds.isna()].index:
    txt = str(df.at[idx, "# Timestamp"]) if "# Timestamp" in df.columns else ""
    m = time_re.search(txt)
    if m:
        h = int(m.group(1))
        mn = int(m.group(2))
        s = int(m.group(3)) if m.group(3) else 0
        seconds.at[idx] = h * 3600 + mn * 60 + s

# Convert to integer second-of-day for exact matching
sec_int = pd.Series(seconds.round().astype("Int64"), index=df.index)

# Define exact intervals (inclusive bounds)
intervals = [
    (12 * 3600 + 18 * 60, 12 * 3600 + 26 * 60),   # 12:18-12:26
    (12 * 3600 + 38 * 60, 13 * 3600 + 0 * 60),    # 12:38-13:00
    (13 * 3600 + 33 * 60, 13 * 3600 + 59 * 60),   # 13:33-13:59
]

in_any_interval = pd.Series(False, index=df.index)
for lo, hi in intervals:
    in_any_interval |= (sec_int.notna() & (sec_int >= lo) & (sec_int <= hi))

filtered = df.loc[in_any_interval].copy()
filtered["__sec_of_day"] = sec_int[in_any_interval]

# Build canonical time-of-day string HH:MM:SS for deduplication
def fmt_hms(s):
    h = int(s // 3600)
    rem = int(s % 3600)
    m = rem // 60
    sec = rem % 60
    return f"{h:02d}:{m:02d}:{sec:02d}"

filtered["__time_of_day"] = filtered["__sec_of_day"].apply(lambda s: fmt_hms(s) if pd.notna(s) else None)

# Deduplicate: keep one row per exact second-level timestamp
filtered_unique = filtered.dropna(subset=["__time_of_day"]).drop_duplicates(subset=["__time_of_day"], keep="first")

# Summary per interval
print("Interval summaries (unique timestamps):")
for lo, hi in intervals:
    mask_i = (filtered_unique["__sec_of_day"] >= lo) & (filtered_unique["__sec_of_day"] <= hi)
    count_i = int(mask_i.sum())
    lo_str = fmt_hms(lo)
    hi_str = fmt_hms(hi)
    print(f"  {lo_str} - {hi_str}: {count_i} rows")

print(f"Total unique timestamps: {len(filtered_unique)}")

# Save results
filtered_unique.to_csv("specific_ships_july.csv", index=False)

filtered_unique

Interval summaries (unique timestamps):
  12:18:00 - 12:26:00: 4 rows
  12:38:00 - 13:00:00: 31 rows
  13:33:00 - 13:59:00: 36 rows
Total unique timestamps: 71


C:\Users\45422\AppData\Local\Temp\ipykernel_16588\3856187632.py:9: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  ts_parsed = pd.to_datetime(df["# Timestamp"], errors="coerce", infer_datetime_format=True)


,# Timestamp,Type of mobile,MMSI,Latitude,Longitude,Navigational status,ROT,SOG,COG,Heading,...,Draught,Destination,ETA,Data source type,A,B,C,D,__sec_of_day,__time_of_day
9642,02/07/2025 12:18:59,AtoN,992191536,55.340950,11.028100,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,44339,12:18:59
9647,02/07/2025 12:19:02,AtoN,992191537,55.342850,11.042867,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,44342,12:19:02
9652,02/07/2025 12:25:00,AtoN,992191536,55.340950,11.028100,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,44700,12:25:00
9657,02/07/2025 12:25:04,AtoN,992191537,55.342850,11.042867,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,44704,12:25:04
9682,02/07/2025 12:43:03,AtoN,992191536,55.340950,11.028100,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,45783,12:43:03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10117,02/07/2025 13:49:56,Class A,354701000,55.347558,11.039695,Under way using engine,0.0,13.6,0.1,1.0,...,10.0,LT KLJ - DK SKA,03/07/2025 05:00:00,AIS,156.0,26.0,21.0,10.0,49796,13:49:56
10123,02/07/2025 13:50:06,Class A,354701000,55.348182,11.039692,Under way using engine,0.0,13.6,359.9,0.0,...,10.0,LT KLJ - DK SKA,03/07/2025 05:00:00,AIS,156.0,26.0,21.0,10.0,49806,13:50:06
10128,02/07/2025 13:50:16,Class A,354701000,55.348802,11.039678,Under way using engine,0.0,13.6,359.6,0.0,...,10.0,LT KLJ - DK SKA,03/07/2025 05:00:00,AIS,156.0,26.0,21.0,10.0,49816,13:50:16
10133,02/07/2025 13:55:15,AtoN,992191536,55.340950,11.028100,Unknown value,NaN,NaN,NaN,NaN,...,NaN,Unknown,NaN,AIS,1.0,1.0,1.0,1.0,50115,13:55:15
